# Notebook 06 — Patient Similarity Network Construction

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Purpose

Construct the first actual **clinical/patient similarity network** from the validated
feature representation in Notebook 05.

### Network definition

- **Node:** one patient
- **Edge:** clinical similarity between two patients
- **Edge weight:** cosine similarity strength
- **No edge:** patients below the selected similarity neighbourhood rule

This is a **clinical/patient similarity network**, not a social network. The dataset does
not contain observed social relationships.

### Critical rules

1. Readmission outcome is never used to create edges.
2. Patient identifiers are used only as node IDs.
3. The similarity representation from Notebook 05 is used.
4. The graph is kept sparse; no dense all-pairs matrix is created.
5. Several `k` values are evaluated before freezing the network configuration.
6. Self-loops and duplicate edges are prohibited.
7. Network diagnostics must be completed before Notebook 07.

## Expected project structure

```text
sna/
├── diabetes+130-us+hospitals+for+years+1999-2008/
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   ├── 02_cleaning.ipynb
│   ├── 03_patient_representation.ipynb
│   ├── 04_split_leakage_check.ipynb
│   ├── 05_features_similarity.ipynb
│   └── 06_similarity_network.ipynb
├── results/
└── figures/
```

### Inputs

- `results/05_X_train.npz`
- `results/05_X_validation.npz`
- `results/05_X_test.npz`
- `results/05_train_feature_patient_ids.csv`
- `results/05_similarity_selection.json`

### Main output

The primary SNA network is constructed on the **training patients**.

Validation/test patients are retained for later evaluation and are not used to tune the
training network topology.

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
import networkx as nx

SEED = 42

K_VALUES = [5, 10, 20, 30]
PRIMARY_K = 10
SIMILARITY_THRESHOLD = 0.70

pd.set_option("display.max_columns", 100)

# ============================================================
# FIND THE SNA PROJECT FOLDER
# ============================================================

possible_roots = [
    Path.cwd(),
    Path.home() / "OneDrive" / "Desktop" / "sna",
    Path.home() / "Desktop" / "sna",
]

PROJECT_ROOT = None

for root in possible_roots:
    if (root / "results" / "05_X_train.npz").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the sna/results folder containing "
        "'05_X_train.npz'. Please check that Notebook 05 "
        "was successfully executed and its outputs exist."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PROJECT FOUND")
print("=" * 70)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print()
print("Notebook 05 outputs found:")
print("05_X_train.npz:",
      (RESULTS_DIR / "05_X_train.npz").exists())
print("05_X_validation.npz:",
      (RESULTS_DIR / "05_X_validation.npz").exists())
print("05_X_test.npz:",
      (RESULTS_DIR / "05_X_test.npz").exists())
print("05_train_feature_patient_ids.csv:",
      (RESULTS_DIR / "05_train_feature_patient_ids.csv").exists())
print("05_similarity_selection.json:",
      (RESULTS_DIR / "05_similarity_selection.json").exists())

PROJECT FOUND
PROJECT_ROOT: C:\Users\Gayatri\OneDrive\Desktop\sna
RESULTS_DIR: C:\Users\Gayatri\OneDrive\Desktop\sna\results

Notebook 05 outputs found:
05_X_train.npz: True
05_X_validation.npz: True
05_X_test.npz: True
05_train_feature_patient_ids.csv: True
05_similarity_selection.json: True


In [4]:
X_train = sparse.load_npz(RESULTS_DIR / "05_X_train.npz").tocsr()
X_valid = sparse.load_npz(RESULTS_DIR / "05_X_validation.npz").tocsr()
X_test = sparse.load_npz(RESULTS_DIR / "05_X_test.npz").tocsr()

train_ids = pd.read_csv(
    RESULTS_DIR / "05_train_feature_patient_ids.csv"
)["patient_nbr"].tolist()

valid_ids = pd.read_csv(
    RESULTS_DIR / "05_validation_feature_patient_ids.csv"
)["patient_nbr"].tolist()

test_ids = pd.read_csv(
    RESULTS_DIR / "05_test_feature_patient_ids.csv"
)["patient_nbr"].tolist()

selection = json.loads(
    (RESULTS_DIR / "05_similarity_selection.json").read_text(encoding="utf-8")
)

print("Train matrix:", X_train.shape)
print("Validation matrix:", X_valid.shape)
print("Test matrix:", X_test.shape)
print("Primary similarity from Notebook 05:", selection["primary_similarity"])

assert selection["primary_similarity"] == "cosine"
assert X_train.shape[0] == len(train_ids)
assert X_valid.shape[0] == len(valid_ids)
assert X_test.shape[0] == len(test_ids)
assert len(set(train_ids)) == len(train_ids)

Train matrix: (50062, 57)
Validation matrix: (10728, 57)
Test matrix: (10728, 57)
Primary similarity from Notebook 05: cosine


# 1. Why the network is built on training patients

The final similarity network is first constructed on the **training population**.

This prevents validation/test patients from influencing the network topology used for
model development and SNA analysis.

Later, validation/test patients can be projected onto the learned representation for
evaluation without changing the frozen training network.

The network is therefore:

`G_train = (V_train, E_train, W_train)`

where each node is a training patient and each edge has a cosine-similarity weight.

In [5]:
# Basic matrix integrity checks
matrix_checks = {
    "train_rows_match_ids": X_train.shape[0] == len(train_ids),
    "train_feature_dimension_positive": X_train.shape[1] > 0,
    "train_finite": np.isfinite(X_train.data).all(),
    "train_ids_unique": len(set(train_ids)) == len(train_ids),
    "validation_ids_unique": len(set(valid_ids)) == len(valid_ids),
    "test_ids_unique": len(set(test_ids)) == len(test_ids),
}

for name, passed in matrix_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(matrix_checks.values())

PASS | train_rows_match_ids
PASS | train_feature_dimension_positive
PASS | train_finite
PASS | train_ids_unique
PASS | validation_ids_unique
PASS | test_ids_unique


# 2. Build k-nearest-neighbour candidate networks

For each candidate `k`, each patient connects to its `k` nearest neighbours by cosine
distance.

Because cosine similarity is symmetric but k-nearest-neighbour selection can be
asymmetric, we use a **mutual-kNN rule**:

An undirected edge `(i,j)` is retained only when `i` is among `j`'s k nearest neighbours
**and** `j` is among `i`'s k nearest neighbours.

The edge weight is the cosine similarity.

This is deliberately conservative and reduces one-sided, weak neighbourhood links.

In [6]:
max_k = max(K_VALUES)

knn = NearestNeighbors(
    n_neighbors=max_k + 1,  # +1 because the closest point is the patient itself
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

knn.fit(X_train)

distances, indices = knn.kneighbors(X_train)

print("kNN distance matrix shape:", distances.shape)
print("kNN index matrix shape:", indices.shape)

assert indices.shape == distances.shape
assert indices.shape[0] == len(train_ids)
assert indices.shape[1] == max_k + 1

kNN distance matrix shape: (50062, 31)
kNN index matrix shape: (50062, 31)


In [8]:
# ============================================================
# Convert each patient's neighbour list into a set
# for mutual-kNN testing.
# ============================================================

neighbor_sets = {}

for i in range(len(train_ids)):

    # Take all returned neighbours and remove the patient itself.
    # We do NOT assume that self is at position 0.
    row_neighbors = np.asarray(indices[i]).ravel()

    nonself_neighbors = [
        int(x)
        for x in row_neighbors
        if int(x) != i
    ]

    # Keep exactly max_k nearest non-self neighbours.
    neighbor_sets[i] = set(nonself_neighbors[:max_k])

print("Example neighbour count:", len(neighbor_sets[0]))

# Every patient should have exactly max_k non-self neighbours.
neighbor_counts = np.array([
    len(v) for v in neighbor_sets.values()
])

print("Minimum neighbour count:", neighbor_counts.min())
print("Maximum neighbour count:", neighbor_counts.max())
print("Mean neighbour count:", neighbor_counts.mean())

assert neighbor_counts.min() == max_k
assert neighbor_counts.max() == max_k

print("NEIGHBOUR SET CHECK: PASS")

Example neighbour count: 30
Minimum neighbour count: 30
Maximum neighbour count: 30
Mean neighbour count: 30.0
NEIGHBOUR SET CHECK: PASS


# 3. Construct candidate graphs and calculate topology diagnostics

For every candidate k, calculate:

- number of nodes;
- number of edges;
- density;
- average degree;
- median degree;
- minimum/maximum degree;
- number of connected components;
- largest connected component fraction;
- number of isolated nodes.

These diagnostics are used to avoid selecting a network that is either almost empty or
almost fully connected.

In [9]:
def build_mutual_knn_graph(k):
    G = nx.Graph()
    G.add_nodes_from(range(len(train_ids)))

    edge_rows = []

    for i in range(len(train_ids)):
        i_neighbors = neighbor_sets[i]

        for j in i_neighbors:
            if i >= j:
                continue

            if i not in neighbor_sets[j]:
                continue

            # Locate cosine distance from i to j.
            row_indices = indices[i, 1:max_k + 1]
            row_distances = distances[i, 1:max_k + 1]

            matches = np.flatnonzero(row_indices == j)
            if len(matches) == 0:
                continue

            d = float(row_distances[matches[0]])
            similarity = float(1.0 - d)

            if not np.isfinite(similarity):
                continue

            # Numerical guard: cosine similarity should be within [-1, 1].
            similarity = float(np.clip(similarity, -1.0, 1.0))

            edge_rows.append({
                "source_row": i,
                "target_row": j,
                "source_patient_nbr": int(train_ids[i]),
                "target_patient_nbr": int(train_ids[j]),
                "similarity": similarity,
                "cosine_distance": d,
                "k": k,
            })

            G.add_edge(i, j, weight=similarity)

    edges = pd.DataFrame(edge_rows)

    return G, edges


candidate_graphs = {}
candidate_edge_tables = []
candidate_network_stats = []

for k in K_VALUES:
    G, edges = build_mutual_knn_graph(k)

    candidate_graphs[k] = G

    n = G.number_of_nodes()
    m = G.number_of_edges()
    degrees = np.array([d for _, d in G.degree()], dtype=float)

    components = list(nx.connected_components(G))
    largest_component = max((len(c) for c in components), default=0)

    candidate_network_stats.append({
        "k": k,
        "nodes": n,
        "edges": m,
        "density": nx.density(G),
        "mean_degree": float(degrees.mean()),
        "median_degree": float(np.median(degrees)),
        "min_degree": int(degrees.min()),
        "max_degree": int(degrees.max()),
        "connected_components": len(components),
        "largest_component_nodes": largest_component,
        "largest_component_fraction": largest_component / n if n else 0,
        "isolated_nodes": int((degrees == 0).sum()),
        "mean_edge_weight": float(edges["similarity"].mean()) if len(edges) else np.nan,
        "median_edge_weight": float(edges["similarity"].median()) if len(edges) else np.nan,
    })

    if not edges.empty:
        candidate_edge_tables.append(edges)

network_stats = pd.DataFrame(candidate_network_stats)

display(network_stats)

,k,nodes,edges,density,mean_degree,median_degree,min_degree,max_degree,connected_components,largest_component_nodes,largest_component_fraction,isolated_nodes,mean_edge_weight,median_edge_weight
0,5,50062,434595,0.000347,17.362271,18.0,0,30,82,49228,0.983341,77,0.918413,0.926002
1,10,50062,434595,0.000347,17.362271,18.0,0,30,82,49228,0.983341,77,0.918413,0.926002
2,20,50062,434595,0.000347,17.362271,18.0,0,30,82,49228,0.983341,77,0.918413,0.926002
3,30,50062,434595,0.000347,17.362271,18.0,0,30,82,49228,0.983341,77,0.918413,0.926002


# 4. Threshold sensitivity

In addition to mutual-kNN, evaluate a similarity threshold on the primary k-neighbour
candidate.

The threshold removes edges whose cosine similarity is below the chosen clinical
similarity cut-off.

This is a sensitivity diagnostic. We do not use readmission outcomes to choose the
threshold.

In [10]:
threshold_G = candidate_graphs[PRIMARY_K].copy()

edges_primary = candidate_edge_tables[
    [K_VALUES.index(PRIMARY_K)]
    if False else 0
] if False else None

# Rebuild the edge table for PRIMARY_K explicitly.
_, primary_edges = build_mutual_knn_graph(PRIMARY_K)

threshold_edges = primary_edges[
    primary_edges["similarity"] >= SIMILARITY_THRESHOLD
].copy()

threshold_G = nx.Graph()
threshold_G.add_nodes_from(range(len(train_ids)))

for row in threshold_edges.itertuples(index=False):
    threshold_G.add_edge(
        int(row.source_row),
        int(row.target_row),
        weight=float(row.similarity)
    )

threshold_stats = {
    "k": PRIMARY_K,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "nodes": threshold_G.number_of_nodes(),
    "edges": threshold_G.number_of_edges(),
    "density": nx.density(threshold_G),
    "mean_degree": float(np.mean([d for _, d in threshold_G.degree()])),
    "isolated_nodes": int(sum(d == 0 for _, d in threshold_G.degree())),
    "connected_components": nx.number_connected_components(threshold_G),
    "largest_component_fraction": (
        max(map(len, nx.connected_components(threshold_G)))
        / threshold_G.number_of_nodes()
    ),
}

print(json.dumps(threshold_stats, indent=2))

{
  "k": 10,
  "similarity_threshold": 0.7,
  "nodes": 50062,
  "edges": 434535,
  "density": 0.0003467744103502105,
  "mean_degree": 17.359873756541887,
  "isolated_nodes": 84,
  "connected_components": 89,
  "largest_component_fraction": 0.9832008309695977
}


# 5. Edge integrity checks

Every edge must satisfy:

- no self-loop;
- unique undirected pair;
- finite similarity;
- similarity within the valid cosine range;
- positive edge strength for the selected graph;
- source/target correspond to valid patient rows.

These checks are hard failures.

In [11]:
def edge_integrity_report(G, edges):
    pair_set = set()
    self_loops = 0
    duplicate_pairs = 0
    invalid_weights = 0
    invalid_rows = 0

    for row in edges.itertuples(index=False):
        i = int(row.source_row)
        j = int(row.target_row)

        if i == j:
            self_loops += 1

        pair = tuple(sorted((i, j)))

        if pair in pair_set:
            duplicate_pairs += 1
        pair_set.add(pair)

        w = float(row.similarity)

        if not np.isfinite(w) or w < -1 or w > 1:
            invalid_weights += 1

        if i < 0 or j < 0 or i >= len(train_ids) or j >= len(train_ids):
            invalid_rows += 1

    return {
        "self_loops": self_loops,
        "duplicate_undirected_pairs": duplicate_pairs,
        "invalid_weights": invalid_weights,
        "invalid_node_rows": invalid_rows,
        "graph_self_loops": nx.number_of_selfloops(G),
    }


primary_G = candidate_graphs[PRIMARY_K]
_, primary_edges = build_mutual_knn_graph(PRIMARY_K)

integrity = edge_integrity_report(primary_G, primary_edges)

print(json.dumps(integrity, indent=2))

assert integrity["self_loops"] == 0
assert integrity["duplicate_undirected_pairs"] == 0
assert integrity["invalid_weights"] == 0
assert integrity["invalid_node_rows"] == 0
assert integrity["graph_self_loops"] == 0

{
  "self_loops": 0,
  "duplicate_undirected_pairs": 0,
  "invalid_weights": 0,
  "invalid_node_rows": 0,
  "graph_self_loops": 0
}


# 6. Select the primary network configuration

The default configuration is:

- cosine similarity;
- mutual kNN;
- `k = 10`.

This is a starting configuration, not an outcome-optimized choice.

The candidate network statistics above must be manually reviewed. The chosen graph should
be sufficiently connected for community/SNA analysis but not so dense that every patient
is effectively connected to everyone.

**Do not tune k using readmission outcomes.**

In [12]:
PRIMARY_NETWORK_RULE = "mutual_knn"
PRIMARY_K = 10

primary_G, primary_edges = build_mutual_knn_graph(PRIMARY_K)

print("Primary network:")
print("Nodes:", primary_G.number_of_nodes())
print("Edges:", primary_G.number_of_edges())
print("Density:", round(nx.density(primary_G), 8))
print("Connected components:", nx.number_connected_components(primary_G))
print(
    "Largest component fraction:",
    round(
        max(map(len, nx.connected_components(primary_G)))
        / primary_G.number_of_nodes(),
        6
    )
)

Primary network:
Nodes: 50062
Edges: 434595
Density: 0.00034682
Connected components: 82
Largest component fraction: 0.983341


# 7. Convert row indices to patient IDs

NetworkX uses row indices internally for efficient construction.

For all saved research outputs, edges are exported using the actual patient identifiers
as `source_patient_nbr` and `target_patient_nbr`.

The identifiers are node labels only; they do not influence similarity.

In [13]:
primary_edges_patient = primary_edges[
    [
        "source_patient_nbr",
        "target_patient_nbr",
        "similarity",
        "cosine_distance",
        "k",
    ]
].copy()

primary_edges_patient = primary_edges_patient.sort_values(
    ["source_patient_nbr", "target_patient_nbr"]
).reset_index(drop=True)

primary_edges_patient.head()

,source_patient_nbr,target_patient_nbr,similarity,cosine_distance,k
0,135,206118,0.909803,0.090197,10
1,135,864882,0.995566,0.004434,10
2,135,1041291,0.899734,0.100266,10
3,135,1275093,0.917639,0.082361,10
4,135,4872942,0.914854,0.085146,10


# 8. Save the primary edge list

The edge list is the most portable representation of the clinical similarity network.

It can later be loaded into NetworkX, Gephi, Cytoscape, or other SNA tooling.

In [14]:
EDGE_OUTPUT = RESULTS_DIR / "06_patient_similarity_edges_k10.csv"
NODE_OUTPUT = RESULTS_DIR / "06_patient_similarity_nodes.csv"
STATS_OUTPUT = RESULTS_DIR / "06_network_candidate_stats.csv"
THRESHOLD_EDGE_OUTPUT = RESULTS_DIR / "06_threshold_edges.csv"

primary_edges_patient.to_csv(EDGE_OUTPUT, index=False)

node_table = pd.DataFrame({
    "patient_nbr": train_ids,
    "node_index": np.arange(len(train_ids)),
    "degree": [primary_G.degree(i) for i in range(len(train_ids))],
})

node_table.to_csv(NODE_OUTPUT, index=False)
network_stats.to_csv(STATS_OUTPUT, index=False)
threshold_edges.to_csv(THRESHOLD_EDGE_OUTPUT, index=False)

print("Saved:")
for p in [EDGE_OUTPUT, NODE_OUTPUT, STATS_OUTPUT, THRESHOLD_EDGE_OUTPUT]:
    print(" -", p)

Saved:
 - C:\Users\Gayatri\OneDrive\Desktop\sna\results\06_patient_similarity_edges_k10.csv
 - C:\Users\Gayatri\OneDrive\Desktop\sna\results\06_patient_similarity_nodes.csv
 - C:\Users\Gayatri\OneDrive\Desktop\sna\results\06_network_candidate_stats.csv
 - C:\Users\Gayatri\OneDrive\Desktop\sna\results\06_threshold_edges.csv


# 9. Network-level diagnostics

Calculate the basic network quantities that Notebook 07 will use:

- weighted degree (strength);
- unweighted degree;
- connected components;
- edge-weight distribution.

Centrality/community analysis itself is deliberately deferred to Notebook 07.

In [15]:
strengths = dict(primary_G.degree(weight="weight"))
degrees = dict(primary_G.degree())

network_diagnostics = pd.DataFrame({
    "patient_nbr": train_ids,
    "degree": [degrees[i] for i in range(len(train_ids))],
    "weighted_degree_strength": [strengths[i] for i in range(len(train_ids))],
})

display(network_diagnostics.describe())

,patient_nbr,degree,weighted_degree_strength
count,5.006200e+04,50062.000000,50062.000000
mean,5.515601e+07,17.362271,15.945739
std,3.946417e+07,7.386576,7.131792
min,1.350000e+02,0.000000,0.000000
25%,2.337060e+07,12.000000,10.264264
50%,4.887879e+07,18.000000,15.919650
75%,8.766438e+07,23.000000,21.730679
max,1.895026e+08,30.000000,29.904983


# 10. Verify train-only network membership

Every node in the primary network must be a training patient.

No validation or test patient can appear in the training network.

In [16]:
train_id_set = set(train_ids)
valid_id_set = set(valid_ids)
test_id_set = set(test_ids)

network_patient_ids = set(primary_edges_patient["source_patient_nbr"]) | set(
    primary_edges_patient["target_patient_nbr"]
)

print("Network edge endpoints:", len(network_patient_ids))
print("Validation IDs appearing in network:",
      len(network_patient_ids & valid_id_set))
print("Test IDs appearing in network:",
      len(network_patient_ids & test_id_set))

assert network_patient_ids.issubset(train_id_set)
assert network_patient_ids.isdisjoint(valid_id_set)
assert network_patient_ids.isdisjoint(test_id_set)

Network edge endpoints: 49985
Validation IDs appearing in network: 0
Test IDs appearing in network: 0


# 11. Freeze network configuration

Save the exact rule used to construct the primary graph.

This prevents later notebooks from silently changing the topology.

In [17]:
CONFIG_OUTPUT = RESULTS_DIR / "06_network_config.json"

network_config = {
    "network_type": "clinical_patient_similarity_network",
    "node_definition": "one training patient",
    "edge_definition": "mutual k-nearest-neighbour clinical similarity",
    "similarity_metric": "cosine",
    "edge_weight": "cosine_similarity",
    "primary_k": int(PRIMARY_K),
    "candidate_k_values": [int(k) for k in K_VALUES],
    "similarity_threshold_sensitivity": float(SIMILARITY_THRESHOLD),
    "target_used_for_network": False,
    "identifier_used_for_similarity": False,
    "validation_test_patients_in_training_network": False,
    "dense_all_pairs_matrix_created": False,
    "seed": int(SEED),
}

CONFIG_OUTPUT.write_text(
    json.dumps(network_config, indent=2),
    encoding="utf-8"
)

print(json.dumps(network_config, indent=2))

{
  "network_type": "clinical_patient_similarity_network",
  "node_definition": "one training patient",
  "edge_definition": "mutual k-nearest-neighbour clinical similarity",
  "similarity_metric": "cosine",
  "edge_weight": "cosine_similarity",
  "primary_k": 10,
  "candidate_k_values": [
    5,
    10,
    20,
    30
  ],
  "similarity_threshold_sensitivity": 0.7,
  "target_used_for_network": false,
  "identifier_used_for_similarity": false,
  "validation_test_patients_in_training_network": false,
  "dense_all_pairs_matrix_created": false,
  "seed": 42
}


# 12. Final checkpoint — Notebook 06

### GO criteria

- Primary network contains the expected training patient population as nodes.
- Edges are based only on cosine clinical similarity.
- Readmission outcome is not used to create edges.
- No validation/test patients enter the training network.
- No self-loops.
- No duplicate undirected edges.
- All edge weights are finite and valid.
- Candidate k values were evaluated.
- Network density/components were calculated.
- Threshold sensitivity was evaluated.
- Edge list and node table are saved.
- Network configuration is frozen and documented.
- No dense all-pairs similarity matrix was created.

### STOP

Stop before Notebook 07 if the selected graph is nearly empty, nearly complete,
contains invalid edges, contains non-training patients, or appears clinically
implausible.

In [18]:
final_checks = {
    "node_count_matches_train_patients": primary_G.number_of_nodes() == len(train_ids),
    "network_edges_exist": primary_G.number_of_edges() > 0,
    "no_self_loops": nx.number_of_selfloops(primary_G) == 0,
    "no_duplicate_undirected_edges": integrity["duplicate_undirected_pairs"] == 0,
    "all_weights_valid": integrity["invalid_weights"] == 0,
    "all_nodes_are_training_patients": (
        network_patient_ids.issubset(train_id_set)
    ),
    "no_validation_nodes": network_patient_ids.isdisjoint(valid_id_set),
    "no_test_nodes": network_patient_ids.isdisjoint(test_id_set),
    "candidate_k_evaluated": set(K_VALUES).issubset(
        set(network_stats["k"].tolist())
    ),
    "threshold_sensitivity_evaluated": True,
    "edge_output_exists": EDGE_OUTPUT.exists(),
    "node_output_exists": NODE_OUTPUT.exists(),
    "stats_output_exists": STATS_OUTPUT.exists(),
    "threshold_output_exists": THRESHOLD_EDGE_OUTPUT.exists(),
    "config_output_exists": CONFIG_OUTPUT.exists(),
    "no_dense_all_pairs_matrix": True,
    "outcome_not_used": network_config["target_used_for_network"] is False,
    "identifier_not_used_for_similarity": network_config["identifier_used_for_similarity"] is False,
}

print("=" * 75)
print("NOTEBOOK 06 — FINAL CHECKPOINT")
print("=" * 75)

for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<6} | {name}")

print("=" * 75)

if all(final_checks.values()):
    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")
else:
    print("OVERALL RESULT: FAIL")
    print("Fix the failed checks before moving to Notebook 07.")

NOTEBOOK 06 — FINAL CHECKPOINT
PASS   | node_count_matches_train_patients
PASS   | network_edges_exist
PASS   | no_self_loops
PASS   | no_duplicate_undirected_edges
PASS   | all_weights_valid
PASS   | all_nodes_are_training_patients
PASS   | no_validation_nodes
PASS   | no_test_nodes
PASS   | candidate_k_evaluated
PASS   | threshold_sensitivity_evaluated
PASS   | edge_output_exists
PASS   | node_output_exists
PASS   | stats_output_exists
PASS   | threshold_output_exists
PASS   | config_output_exists
PASS   | no_dense_all_pairs_matrix
PASS   | outcome_not_used
PASS   | identifier_not_used_for_similarity
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.


In [19]:
summary = {
    "notebook": "06_similarity_network",
    "seed": int(SEED),
    "network_type": "clinical_patient_similarity_network",
    "similarity_metric": "cosine",
    "network_rule": PRIMARY_NETWORK_RULE,
    "primary_k": int(PRIMARY_K),
    "similarity_threshold_sensitivity": float(SIMILARITY_THRESHOLD),
    "train_nodes": int(primary_G.number_of_nodes()),
    "primary_edges": int(primary_G.number_of_edges()),
    "primary_density": float(nx.density(primary_G)),
    "connected_components": int(nx.number_connected_components(primary_G)),
    "largest_component_fraction": float(
        max(map(len, nx.connected_components(primary_G)))
        / primary_G.number_of_nodes()
    ),
    "mean_degree": float(np.mean([d for _, d in primary_G.degree()])),
    "median_degree": float(np.median([d for _, d in primary_G.degree()])),
    "mean_edge_weight": float(primary_edges_patient["similarity"].mean()),
    "median_edge_weight": float(primary_edges_patient["similarity"].median()),
    "target_used_for_network": False,
    "dense_all_pairs_matrix_created": False,
    "final_checks": {str(k): bool(v) for k, v in final_checks.items()},
}

SUMMARY_OUTPUT = RESULTS_DIR / "06_network_summary.json"
SUMMARY_OUTPUT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Summary saved:", SUMMARY_OUTPUT)

Summary saved: C:\Users\Gayatri\OneDrive\Desktop\sna\results\06_network_summary.json
